#  NutriSense AI — ConvNeXt-Small Food Image Classifier

This notebook trains a **ConvNeXt-Small** model on the NutriSense food image dataset (239 classes).  
It is designed to run on **Kaggle with multiple GPUs** (using PyTorch `DataParallel`).

### Pipeline
1. **Junk File Cleanup** — Remove non-image files (.txt, .cms, .svg, etc.)
2. **Corrupt Image Detection** — Verify & remove broken images
3. **Exploratory Data Analysis** — Class distribution analysis
4. **Data Preprocessing** — Augmentation, normalization, class-weighted sampling
5. **Model Setup** — ConvNeXt-Small with pretrained weights, custom classifier head
6. **Multi-GPU Training** — With mixed precision, cosine annealing, early stopping
7. **Evaluation** — Accuracy, confusion matrix, classification report
8. **Inference** — Predict folder name (class) from a single image

## 1. Install & Import Dependencies

In [ ]:
!pip install -q timm

import os
import sys
import glob
import shutil
import random
import warnings
import time
from pathlib import Path
from collections import Counter
from PIL import Image, ImageFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as T
from torchvision.datasets import ImageFolder

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm.auto import tqdm

# Allow loading truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 2. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Adjust paths and hyperparameters here
# ============================================================

# Dataset path (change this for Kaggle)
# On Kaggle: '/kaggle/input/indian-food-images/Images'
# Locally:   r'd:\Projects\NutriSense-AI\Dataset\Images'
DATASET_DIR = '/kaggle/input/indian-food-images/Images'  # <-- Change for your environment

# Training hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 64          # Per-GPU batch size will be BATCH_SIZE // num_gpus
NUM_WORKERS = 4
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
PATIENCE = 7             # Early stopping patience
MIN_DELTA = 0.001        # Minimum improvement for early stopping

# Train/Val/Test split ratios
VAL_RATIO = 0.15
TEST_RATIO = 0.10

# Output paths
MODEL_SAVE_PATH = 'nutrisense_convnext_small_best.pth'
FINAL_MODEL_SAVE_PATH = 'nutrisense_convnext_small_final.pth'
CLASS_NAMES_PATH = 'nutrisense_class_names.npy'

# Valid image extensions
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff', '.tif'}

print("Configuration loaded.")
print(f"  Dataset: {DATASET_DIR}")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Best model path:  {MODEL_SAVE_PATH}")
print(f"  Final model path: {FINAL_MODEL_SAVE_PATH}")

## 3. Junk File Cleanup

Remove all non-image files (`.txt`, `.cms`, `.svg`, etc.) from the dataset to prevent errors during loading.

In [ ]:
def cleanup_junk_files(dataset_dir):
    """Identify and remove non-image files from the dataset directory.
    Handles Kaggle's read-only /kaggle/input/ gracefully."""
    junk_found = []
    removed = []
    
    for root, dirs, files in os.walk(dataset_dir):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext not in VALID_EXTENSIONS:
                filepath = os.path.join(root, f)
                junk_found.append(filepath)
                try:
                    os.remove(filepath)
                    removed.append(filepath)
                except OSError:
                    pass  # Read-only filesystem (e.g., Kaggle input)
    
    if junk_found:
        ext_counts = Counter(os.path.splitext(f)[1].lower() for f in junk_found)
        if removed:
            print(f"Removed {len(removed)} junk file(s):")
        else:
            print(f"Found {len(junk_found)} junk file(s) (read-only FS, will be filtered during loading):")
        for ext, count in ext_counts.items():
            print(f"   {ext}: {count} file(s)")
        print()
        for f in junk_found[:15]:
            print(f"   {'Removed' if f in removed else 'Skipped'}: {f}")
        if len(junk_found) > 15:
            print(f"   ... and {len(junk_found) - 15} more")
    else:
        print("No junk files found. Dataset is clean!")
    
    return set(junk_found)

junk_files_set = cleanup_junk_files(DATASET_DIR)

## 4. Corrupt Image Detection & Removal

Scan all images, remove any that cannot be opened or are corrupted.

In [ ]:
def verify_and_clean_images(dataset_dir, junk_files):
    """Verify all images can be opened and are valid. 
    Returns set of corrupt file paths to exclude during loading."""
    corrupt_files = []
    total_checked = 0
    
    class_dirs = sorted([d for d in os.listdir(dataset_dir) 
                         if os.path.isdir(os.path.join(dataset_dir, d))])
    
    for class_name in tqdm(class_dirs, desc="Verifying images"):
        class_path = os.path.join(dataset_dir, class_name)
        for fname in os.listdir(class_path):
            fpath = os.path.join(class_path, fname)
            if not os.path.isfile(fpath) or fpath in junk_files:
                continue
            total_checked += 1
            try:
                with Image.open(fpath) as img:
                    img.verify()
                with Image.open(fpath) as img:
                    img.load()
                    img.convert('RGB')
            except Exception as e:
                corrupt_files.append((fpath, str(e)))
                try:
                    os.remove(fpath)
                except OSError:
                    pass  # Read-only filesystem
    
    print(f"\nChecked {total_checked} images.")
    if corrupt_files:
        print(f"Found {len(corrupt_files)} corrupt image(s) (excluded from training):")
        for fp, err in corrupt_files[:15]:
            print(f"   {fp} — {err}")
    else:
        print("All images are valid!")
    
    return set(fp for fp, _ in corrupt_files)

corrupt_files_set = verify_and_clean_images(DATASET_DIR, junk_files_set)

## 5. Remove Empty Folders

After cleanup, remove any class folders that have 0 images remaining.

In [ ]:
def check_empty_class_dirs(dataset_dir, junk_files, corrupt_files):
    """Check for class directories with no valid images."""
    empty_dirs = []
    exclude_set = junk_files | corrupt_files
    
    for d in sorted(os.listdir(dataset_dir)):
        dpath = os.path.join(dataset_dir, d)
        if os.path.isdir(dpath):
            valid_files = [
                f for f in os.listdir(dpath)
                if os.path.isfile(os.path.join(dpath, f))
                and os.path.join(dpath, f) not in exclude_set
                and os.path.splitext(f)[1].lower() in VALID_EXTENSIONS
            ]
            if len(valid_files) == 0:
                empty_dirs.append(d)
                try:
                    shutil.rmtree(dpath)
                    print(f"   Removed empty class folder: {d}")
                except OSError:
                    print(f"   Empty class (read-only, will be skipped): {d}")
    
    if not empty_dirs:
        print("No empty class folders found.")
    else:
        print(f"\nFound {len(empty_dirs)} empty class folder(s).")
    return set(empty_dirs)

empty_class_dirs = check_empty_class_dirs(DATASET_DIR, junk_files_set, corrupt_files_set)

## 6. Exploratory Data Analysis

In [ ]:
def get_class_distribution(dataset_dir):
    """Get the number of images per class."""
    class_counts = {}
    class_dirs = sorted([d for d in os.listdir(dataset_dir)
                         if os.path.isdir(os.path.join(dataset_dir, d))])
    
    for class_name in class_dirs:
        class_path = os.path.join(dataset_dir, class_name)
        count = len([f for f in os.listdir(class_path) 
                     if os.path.isfile(os.path.join(class_path, f))])
        class_counts[class_name] = count
    
    return class_counts

class_counts = get_class_distribution(DATASET_DIR)
num_classes = len(class_counts)

print(f"Total classes: {num_classes}")
print(f"Total images: {sum(class_counts.values())}")
print(f"\nClass distribution stats:")
counts = list(class_counts.values())
print(f"  Min images/class:  {min(counts)} ({min(class_counts, key=class_counts.get)})")
print(f"  Max images/class:  {max(counts)} ({max(class_counts, key=class_counts.get)})")
print(f"  Mean images/class: {np.mean(counts):.1f}")
print(f"  Median:            {np.median(counts):.1f}")
print(f"  Std dev:           {np.std(counts):.1f}")

# Imbalance ratio
imbalance_ratio = max(counts) / min(counts)
print(f"  Imbalance ratio:   {imbalance_ratio:.1f}x")

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Histogram of class sizes
axes[0].hist(counts, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Images')
axes[0].set_ylabel('Number of Classes')
axes[0].set_title('Distribution of Images per Class')
axes[0].axvline(np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.0f}')
axes[0].legend()

# Top 20 and Bottom 20 classes
sorted_counts = sorted(class_counts.items(), key=lambda x: x[1])
bottom_20 = sorted_counts[:20]
top_20 = sorted_counts[-20:]

combined = bottom_20 + top_20
names = [c[0] for c in combined]
vals = [c[1] for c in combined]
colors = ['#e74c3c'] * 20 + ['#2ecc71'] * 20

axes[1].barh(names, vals, color=colors)
axes[1].set_xlabel('Number of Images')
axes[1].set_title('Bottom 20 (red) & Top 20 (green) Classes')

plt.tight_layout()
plt.show()

## 7. Build File Lists & Stratified Splits

Create train/val/test splits with **stratified sampling** to maintain class proportions.

In [ ]:
def build_file_list(dataset_dir, exclude_files=None, exclude_dirs=None):
    """Build a list of (image_path, class_index) tuples.
    Filters by valid extensions and excludes known junk/corrupt files."""
    exclude_files = exclude_files or set()
    exclude_dirs = exclude_dirs or set()
    
    class_dirs = sorted([
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d)) and d not in exclude_dirs
    ])
    class_to_idx = {cls: idx for idx, cls in enumerate(class_dirs)}
    
    file_list = []
    labels = []
    skipped = 0
    for class_name in class_dirs:
        class_path = os.path.join(dataset_dir, class_name)
        for fname in os.listdir(class_path):
            fpath = os.path.join(class_path, fname)
            ext = os.path.splitext(fname)[1].lower()
            # Skip: not a file, not a valid image extension, or in exclude list
            if not os.path.isfile(fpath) or ext not in VALID_EXTENSIONS or fpath in exclude_files:
                skipped += 1
                continue
            file_list.append(fpath)
            labels.append(class_to_idx[class_name])
    
    return file_list, labels, class_dirs, class_to_idx, skipped

# Combine all exclusions
all_excluded = junk_files_set | corrupt_files_set
all_files, all_labels, class_names, class_to_idx, num_skipped = build_file_list(
    DATASET_DIR, exclude_files=all_excluded, exclude_dirs=empty_class_dirs
)

# Stratified train/val/test split
# First split: train+val vs test
train_val_files, test_files, train_val_labels, test_labels = train_test_split(
    all_files, all_labels, test_size=TEST_RATIO, stratify=all_labels, random_state=SEED
)

# Second split: train vs val
val_ratio_adjusted = VAL_RATIO / (1 - TEST_RATIO)  # Adjust ratio for remaining data
train_files, val_files, train_labels, val_labels = train_test_split(
    train_val_files, train_val_labels, test_size=val_ratio_adjusted, 
    stratify=train_val_labels, random_state=SEED
)

print(f"Dataset splits (skipped {num_skipped} invalid files):")
print(f"  Train: {len(train_files):>6} images ({len(train_files)/len(all_files)*100:.1f}%)")
print(f"  Val:   {len(val_files):>6} images ({len(val_files)/len(all_files)*100:.1f}%)")
print(f"  Test:  {len(test_files):>6} images ({len(test_files)/len(all_files)*100:.1f}%)")
print(f"  Total: {len(all_files):>6} images across {len(class_names)} classes")

# Save class names for inference
np.save(CLASS_NAMES_PATH, np.array(class_names))
print(f"\n Saved {len(class_names)} class names to {CLASS_NAMES_PATH}")

## 8. Dataset & Data Augmentation

**Anti-bias measures:**
- Heavy augmentation on training set (random crop, flip, rotation, color jitter, erasing)
- Weighted random sampling to handle class imbalance
- Label smoothing in loss function
- ImageNet normalization (ConvNeXt pretrained stats)

In [ ]:
# ImageNet normalization (used by ConvNeXt pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transforms with strong augmentation
train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.05),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.RandomGrayscale(p=0.05),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),  # Cutout-style augmentation
])

# Validation/Test transforms (no augmentation)
val_transform = T.Compose([
    T.Resize(int(IMG_SIZE * 1.14)),   # Resize to 256 if IMG_SIZE=224
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("\u2705 Transforms defined.")
print(f"\nTraining augmentations:")
for t in train_transform.transforms:
    print(f"  \u2022 {t.__class__.__name__}")

In [ ]:
class FoodDataset(Dataset):
    """Custom Dataset for NutriSense food images."""
    
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            # Fallback: return a blank image if corrupted (shouldn't happen after cleanup)
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Create datasets
train_dataset = FoodDataset(train_files, train_labels, transform=train_transform)
val_dataset = FoodDataset(val_files, val_labels, transform=val_transform)
test_dataset = FoodDataset(test_files, test_labels, transform=val_transform)

print(f"\u2705 Datasets created:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")

## 9. Weighted Random Sampler (Class Imbalance Handling)

Ensures underrepresented classes are sampled more frequently → reduces bias.

In [ ]:
def create_weighted_sampler(labels):
    """Create a WeightedRandomSampler to handle class imbalance."""
    class_count = Counter(labels)
    total = len(labels)
    
    # Inverse frequency weighting
    class_weights = {cls: total / count for cls, count in class_count.items()}
    
    # Assign weight to each sample
    sample_weights = [class_weights[label] for label in labels]
    sample_weights = torch.DoubleTensor(sample_weights)
    
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    
    return sampler, class_weights

train_sampler, class_weights_dict = create_weighted_sampler(train_labels)

print("\u2705 Weighted sampler created.")
print(f"   Unique classes in training set: {len(set(train_labels))}")
print(f"   Weight range: {min(class_weights_dict.values()):.2f} - {max(class_weights_dict.values()):.2f}")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f" DataLoaders created.")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

# Verify a batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"\nSample batch shape: {sample_batch.shape}")
print(f"Sample labels shape: {sample_labels.shape}")

## 10. Visualize Augmented Samples

In [ ]:
def show_augmented_samples(dataset, class_names, n=8):
    """Display augmented training samples."""
    fig, axes = plt.subplots(2, n//2, figsize=(16, 7))
    axes = axes.flatten()
    
    indices = random.sample(range(len(dataset)), n)
    
    for ax, idx in zip(axes, indices):
        img, label = dataset[idx]
        # Denormalize
        img = img.clone()
        for c in range(3):
            img[c] = img[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
        img = img.clamp(0, 1)
        img = img.permute(1, 2, 0).numpy()
        
        ax.imshow(img)
        ax.set_title(class_names[label], fontsize=9)
        ax.axis('off')
    
    plt.suptitle('Augmented Training Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_augmented_samples(train_dataset, class_names)

## 11. Build ConvNeXt-Small Model

- Pretrained on ImageNet-1K (via `timm`)
- Custom classifier head with Dropout for regularization
- Multi-GPU support with `DataParallel`

In [ ]:
def build_model(num_classes, pretrained=True):
    """Build ConvNeXt-Small model with custom classification head."""
    
    # Load pretrained ConvNeXt-Small
    model = timm.create_model('convnext_small.fb_in22k_ft_in1k', pretrained=pretrained)
    
    # Get the number of features from the classifier
    in_features = model.head.fc.in_features
    
    # Replace the classifier head with a custom one
    model.head.fc = nn.Sequential(
        nn.LayerNorm(in_features),
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.GELU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
    
    return model

# Build model
model = build_model(num_classes=num_classes, pretrained=True)

# Move to GPU(s)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.device_count() > 1:
    print(f"\u2705 Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)
else:
    print(f"\u2705 Using single device: {device}")

model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: ConvNeXt-Small")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"  Output classes:       {num_classes}")

## 12. Loss, Optimizer & Scheduler

In [ ]:
# Loss function with label smoothing (reduces overconfidence / bias)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# AdamW optimizer (weight decay decoupled from gradients)
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# Cosine Annealing with Warm Restarts for better convergence
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,        # Restart every 10 epochs
    T_mult=2,      # Double the restart period after each restart
    eta_min=1e-7   # Minimum learning rate
)

# Mixed precision scaler
scaler = GradScaler()

print("\u2705 Training components ready:")
print(f"  Loss:      CrossEntropyLoss (label_smoothing={LABEL_SMOOTHING})")
print(f"  Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"  Scheduler: CosineAnnealingWarmRestarts (T_0=10, T_mult=2)")
print(f"  Precision: Mixed (FP16)")

## 13. Training & Validation Loop

Features:
- **Mixed Precision Training** (FP16) for faster training on GPUs
- **Epoch number printed** at each epoch
- **Early stopping** to prevent overfitting
- **Best model checkpoint** saved based on validation accuracy
- **Gradient clipping** for training stability

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_acc):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch, total_epochs):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Train]", 
                leave=True, ncols=120)
    
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        
        # Gradient clipping for stability
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        # Metrics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{running_loss/total:.4f}",
            'acc': f"{100.*correct/total:.2f}%"
        })
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device, epoch, total_epochs):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Val]  ", 
                leave=True, ncols=120)
    
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f"{running_loss/total:.4f}",
            'acc': f"{100.*correct/total:.2f}%"
        })
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

print("\u2705 Training functions defined.")

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

print("=" * 80)
print(f"  TRAINING ConvNeXt-Small | {num_classes} classes | {EPOCHS} epochs")
print(f"  GPUs: {torch.cuda.device_count()} | Batch size: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print("=" * 80)

training_start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\n{'='*80}")
    print(f"  EPOCH {epoch}/{EPOCHS}  |  Learning Rate: {current_lr:.2e}")
    print(f"{'='*80}")
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch, EPOCHS
    )
    
    # Validate
    val_loss, val_acc = validate(
        model, val_loader, criterion, device, epoch, EPOCHS
    )
    
    # Step scheduler
    scheduler.step(epoch)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    epoch_time = time.time() - epoch_start
    
    # Print epoch summary
    print(f"\n  Epoch {epoch}/{EPOCHS} Summary:")
    print(f"    Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc*100:.2f}%")
    print(f"    Val   Loss: {val_loss:.4f}  |  Val   Acc: {val_acc*100:.2f}%")
    print(f"    Time: {epoch_time:.1f}s  |  LR: {current_lr:.2e}")
    
    # Save best model (based on validation accuracy)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Handle DataParallel state dict
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'num_classes': num_classes,
            'class_names': class_names,
            'img_size': IMG_SIZE,
        }, MODEL_SAVE_PATH)
        print(f"     New best model saved! Val Acc: {val_acc*100:.2f}%")
    
    # Early stopping check
    early_stopping(val_acc)
    if early_stopping.early_stop:
        print(f"\n  Early stopping triggered at epoch {epoch}!")
        print(f"    No improvement for {PATIENCE} consecutive epochs.")
        break

# ============================================================
# Save FINAL model (last epoch state, regardless of val accuracy)
# ============================================================
final_epoch = epoch
final_model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save({
    'epoch': final_epoch,
    'model_state_dict': final_model_state,
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'val_acc': val_acc,
    'val_loss': val_loss,
    'train_acc': train_acc,
    'train_loss': train_loss,
    'num_classes': num_classes,
    'class_names': class_names,
    'img_size': IMG_SIZE,
    'history': history,
}, FINAL_MODEL_SAVE_PATH)

total_time = time.time() - training_start_time
print(f"\n{'='*80}")
print(f"  TRAINING COMPLETE")
print(f"  Total time: {total_time/60:.1f} minutes")
print(f"  Best Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"  Final Epoch: {final_epoch}  |  Final Val Acc: {val_acc*100:.2f}%")
print(f"  Best  model saved to: {MODEL_SAVE_PATH}")
print(f"  Final model saved to: {FINAL_MODEL_SAVE_PATH}")
print(f"{'='*80}")

## 14. Training Curves

In [ ]:
def plot_training_history(history):
    """Plot training & validation loss, accuracy, and learning rate."""
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Loss
    axes[0].plot(epochs_range, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
    axes[0].plot(epochs_range, history['val_loss'], 'r-o', markersize=3, label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', markersize=3, label='Train Acc')
    axes[1].plot(epochs_range, [a*100 for a in history['val_acc']], 'r-o', markersize=3, label='Val Acc')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training & Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[2].plot(epochs_range, history['lr'], 'g-o', markersize=3)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Learning Rate Schedule')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('NutriSense ConvNeXt-Small Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 15. Evaluate on Test Set

In [ ]:
# Load best model
print("Loading best model checkpoint...")
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)

# Rebuild model (without DataParallel for clean evaluation)
eval_model = build_model(num_classes=num_classes, pretrained=False)
eval_model.load_state_dict(checkpoint['model_state_dict'])

if torch.cuda.device_count() > 1:
    eval_model = nn.DataParallel(eval_model)

eval_model = eval_model.to(device)
eval_model.eval()

print(f"\u2705 Loaded best model from epoch {checkpoint['epoch']} (Val Acc: {checkpoint['val_acc']*100:.2f}%)")

# Evaluate on test set
all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)
        
        with autocast():
            outputs = eval_model(images)
        
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

test_acc = accuracy_score(all_targets, all_preds)
print(f"\n{'='*60}")
print(f"  TEST ACCURACY: {test_acc*100:.2f}%")
print(f"{'='*60}")

In [ ]:
# Classification Report (Top-level metrics per class)
print("\nClassification Report (macro & weighted averages):")
print("=" * 60)
report = classification_report(
    all_targets, all_preds, 
    target_names=class_names, 
    output_dict=True
)

# Print summary metrics
print(f"  Macro Avg    - Precision: {report['macro avg']['precision']:.4f}  "
      f"Recall: {report['macro avg']['recall']:.4f}  "
      f"F1: {report['macro avg']['f1-score']:.4f}")
print(f"  Weighted Avg - Precision: {report['weighted avg']['precision']:.4f}  "
      f"Recall: {report['weighted avg']['recall']:.4f}  "
      f"F1: {report['weighted avg']['f1-score']:.4f}")

# Show worst performing classes to identify bias
class_f1 = {name: report[name]['f1-score'] for name in class_names if name in report}
sorted_f1 = sorted(class_f1.items(), key=lambda x: x[1])

print(f"\n\u26a0\ufe0f  Bottom 10 Classes by F1-Score (potential bias areas):")
for cls_name, f1 in sorted_f1[:10]:
    print(f"    {cls_name:<30} F1: {f1:.4f}  "
          f"(Precision: {report[cls_name]['precision']:.4f}, "
          f"Recall: {report[cls_name]['recall']:.4f})")

print(f"\n\u2b50 Top 10 Classes by F1-Score:")
for cls_name, f1 in sorted_f1[-10:]:
    print(f"    {cls_name:<30} F1: {f1:.4f}")

In [ ]:
# Confusion Matrix (subset for visibility)
cm = confusion_matrix(all_targets, all_preds)

# Show top-N confused pairs
print("\nMost Confused Class Pairs:")
print("=" * 60)

# Find off-diagonal maximums
confused_pairs = []
for i in range(len(cm)):
    for j in range(len(cm)):
        if i != j and cm[i][j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i][j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confused_pairs[:15]:
    print(f"  {true_cls:<25} \u2192 predicted as {pred_cls:<25} ({count} times)")

# Plot confusion matrix for top-25 most common classes
top_25_indices = np.argsort([class_counts[c] for c in class_names])[-25:]
cm_subset = cm[np.ix_(top_25_indices, top_25_indices)]
subset_names = [class_names[i] for i in top_25_indices]

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_subset, annot=True, fmt='d', cmap='Blues',
            xticklabels=subset_names, yticklabels=subset_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Top 25 Classes by Size)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 16. Inference — Predict Food Class from Image

This function loads a single image and predicts the **folder name** (food class).

In [ ]:
def predict_image(image_path, model, class_names, device, transform, top_k=5):
    """
    Predict the food class (folder name) for a single image.
    
    Args:
        image_path: Path to the image file
        model: Trained model
        class_names: List of class names (folder names)
        device: torch device
        transform: Validation/inference transforms
        top_k: Number of top predictions to return
    
    Returns:
        predicted_class (str): The predicted folder name
        confidence (float): Confidence score
        top_k_results: List of (class_name, confidence) tuples
    """
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        with autocast():
            output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
    
    # Get top-k predictions
    top_probs, top_indices = probabilities.topk(top_k, dim=1)
    top_probs = top_probs.squeeze().cpu().numpy()
    top_indices = top_indices.squeeze().cpu().numpy()
    
    top_k_results = [(class_names[idx], prob) for idx, prob in zip(top_indices, top_probs)]
    predicted_class = top_k_results[0][0]
    confidence = top_k_results[0][1]
    
    return predicted_class, confidence, top_k_results


def predict_and_display(image_path, model, class_names, device, transform, top_k=5):
    """Predict and display the result with the image."""
    predicted_class, confidence, top_k_results = predict_image(
        image_path, model, class_names, device, transform, top_k
    )
    
    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.5]})
    
    # Show image
    img = Image.open(image_path).convert('RGB')
    ax1.imshow(img)
    ax1.set_title(f'Predicted: {predicted_class}\nConfidence: {confidence:.1%}', 
                  fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # Show top-k predictions
    names = [r[0] for r in top_k_results]
    probs = [r[1] for r in top_k_results]
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(names))]
    
    bars = ax2.barh(range(len(names)), probs, color=colors)
    ax2.set_yticks(range(len(names)))
    ax2.set_yticklabels(names)
    ax2.set_xlabel('Confidence')
    ax2.set_title(f'Top-{top_k} Predictions')
    ax2.set_xlim(0, 1)
    ax2.invert_yaxis()
    
    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{prob:.1%}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n\u2705 Predicted Food: {predicted_class}")
    print(f"   Confidence: {confidence:.2%}")
    return predicted_class, confidence

print("\u2705 Inference functions ready.")
print("\nUsage:")
print('  predicted_class, confidence = predict_and_display("path/to/image.jpg", eval_model, class_names, device, val_transform)')

In [ ]:
# Demo: Predict on random test images
print("Predicting on 6 random test images...\n")

sample_indices = random.sample(range(len(test_files)), min(6, len(test_files)))

for idx in sample_indices:
    img_path = test_files[idx]
    true_label = class_names[test_labels[idx]]
    
    pred_class, conf, _ = predict_image(img_path, eval_model, class_names, device, val_transform)
    
    status = "\u2705" if pred_class == true_label else "\u274c"
    print(f"  {status} True: {true_label:<25} | Predicted: {pred_class:<25} | Conf: {conf:.2%}")

## 17. Export Model Info

Save everything needed for production inference.

In [ ]:
# Final model summary
print("=" * 60)
print("  NUTRISENSE AI - ConvNeXt-Small Model Summary")
print("=" * 60)
print(f"  Model:           ConvNeXt-Small (pretrained: ImageNet-22K → 1K)")
print(f"  Classes:         {num_classes}")
print(f"  Image Size:      {IMG_SIZE}x{IMG_SIZE}")
print(f"  Best Val Acc:    {best_val_acc*100:.2f}%")
print(f"  Test Acc:        {test_acc*100:.2f}%")
print(f"  Parameters:      {total_params:,}")
print(f"\n  Saved Models:")
print(f"    Best  model: {MODEL_SAVE_PATH}")
print(f"    Final model: {FINAL_MODEL_SAVE_PATH}")
print(f"    Class Names: {CLASS_NAMES_PATH}")

# Show file sizes
import os as _os
for path, label in [(MODEL_SAVE_PATH, "Best"), (FINAL_MODEL_SAVE_PATH, "Final")]:
    if _os.path.exists(path):
        size_mb = _os.path.getsize(path) / (1024 * 1024)
        print(f"    {label} model size: {size_mb:.1f} MB")

print(f"\n  Normalization:")
print(f"    Mean: {IMAGENET_MEAN}")
print(f"    Std:  {IMAGENET_STD}")
print(f"\n  Anti-Bias Measures Applied:")
print(f"    • Junk file removal")
print(f"    • Corrupt image removal")
print(f"    • Stratified train/val/test splits")
print(f"    • Weighted random sampling (class imbalance)")
print(f"    • Heavy data augmentation")
print(f"    • Label smoothing ({LABEL_SMOOTHING})")
print(f"    • Dropout regularization")
print(f"    • Gradient clipping")
print(f"    • Early stopping")
print("=" * 60)
print("   Model is ready for NutriSense AI deployment!")
print("=" * 60)